# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [22]:
df = pd.read_csv("data/AviationData.csv", encoding="cp1252", low_memory=False)
print(df.shape)
print(df.dtypes)
print(df.isna().sum().sort_values(ascending=False).head(15))
print(df.describe(include="all").T.head(10).to_string())

(88889, 31)
Event.Id                   object
Investigation.Type         object
Accident.Number            object
Event.Date                 object
Location                   object
Country                    object
Latitude                   object
Longitude                  object
Airport.Code               object
Airport.Name               object
Injury.Severity            object
Aircraft.damage            object
Aircraft.Category          object
Registration.Number        object
Make                       object
Model                      object
Amateur.Built              object
Number.of.Engines         float64
Engine.Type                object
FAR.Description            object
Schedule                   object
Purpose.of.flight          object
Air.carrier                object
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition          object
Broad.phase.of.flight      object
Re

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [23]:
df = df.copy()
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")
df = df[
    (df["Event.Date"].dt.year >= 1983)
    & (
        df["Aircraft.Category"].isna()
        | df["Aircraft.Category"].fillna("").str.lower().eq("airplane")
    )
    & (df["Amateur.Built"].fillna("").str.lower() != "yes")

].copy()print(df.shape)
df = df.dropna(subset=["Make", "Model"]).reset_index(drop=True)

SyntaxError: invalid syntax (2817661810.py, line 11)

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]
for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
df["Total.Passengers.Est"] = df[injury_cols].sum(axis=1)
df["Serious.Fatal.Fraction"] = np.where(
    df["Total.Passengers.Est"] > 0,
    (df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]) / df["Total.Passengers.Est"],
    np.nan,
)
df["Fatal.Fraction"] = np.where(
    df["Total.Passengers.Est"] > 0,
    df["Total.Fatal.Injuries"] / df["Total.Passengers.Est"],
    np.nan,
)
df[["Total.Passengers.Est", "Serious.Fatal.Fraction", "Fatal.Fraction"]].head()

,Total.Passengers.Est,Serious.Fatal.Fraction,Fatal.Fraction
0,588.0,0.0,0.0
1,588.0,0.0,0.0
2,2.0,1.0,0.5
3,5.0,0.2,0.2
4,289.0,0.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
df["Aircraft.Damage.Clean"] = df["Aircraft.damage"].fillna("Unknown").astype(str).str.strip().str.title()
df["Destroyed.Flag"] = df["Aircraft.Damage.Clean"].eq("Destroyed").astype(int)
df[["Aircraft.damage", "Aircraft.Damage.Clean", "Destroyed.Flag"]].head(10)

,Aircraft.damage,Aircraft.Damage.Clean,Destroyed.Flag
0,Minor,Minor,0
1,Minor,Minor,0
2,Destroyed,Destroyed,1
3,NaN,Unknown,0
4,Minor,Minor,0
5,Destroyed,Destroyed,1
6,NaN,Unknown,0
7,Minor,Minor,0
8,Substantial,Substantial,0
9,Substantial,Substantial,0


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [ ]:
make_map = {
    "CESSNA": "Cessna",
    "PIPER": "Piper",
    "BOEING": "Boeing",
    "BEECH": "Beech",
    "BELL": "Bell",
    "MCDONNELL DOUGLAS": "McDonnell Douglas",
    "DOUGLAS": "Douglas",
    "SIKORSKY": "Sikorsky",
    "AIRBUS": "Airbus",
    "DE HAVILLAND": "De Havilland",
}
df["Make.Clean"] = df["Make"].fillna("Unknown").astype(str).str.strip()
df["Make.Clean"] = df["Make.Clean"].str.upper().replace(make_map)
df["Make.Clean"] = df["Make.Clean"].replace({"UNKNOWN": "Unknown"})
make_counts = df["Make.Clean"].value_counts()
df = df[df["Make.Clean"].isin(make_counts[make_counts >= 50].index)].copy()
df["Make.Clean"].value_counts().head(20)

Make.Clean
Cessna                7144
Piper                 3991
Beech                 1432
Boeing                1267
MOONEY                 363
Airbus                 244
CIRRUS DESIGN CORP     220
BELLANCA               219
AIR TRACTOR INC        219
MAULE                  215
AIR TRACTOR            206
AERONCA                200
CHAMPION               158
EMBRAER                152
GRUMMAN                147
LUSCOMBE               141
CIRRUS                 137
STINSON                129
McDonnell Douglas      108
NORTH AMERICAN         106
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
df["Model.Clean"] = df["Model"].fillna("Unknown").astype(str).str.strip()
df["Plane.Type"] = df["Make.Clean"] + " " + df["Model.Clean"]
print(df[["Make.Clean", "Model.Clean", "Plane.Type"]].head(10).to_string())
print(df["Plane.Type"].nunique())

      Make.Clean Model.Clean          Plane.Type
1         Boeing         747          Boeing 747
2          Piper   PA-28-140     Piper PA-28-140
3   De Havilland       DHC-6  De Havilland DHC-6
7         Boeing     727-200      Boeing 727-200
8          Beech         C35           Beech C35
9         Cessna        180K         Cessna 180K
10         Beech          99            Beech 99
11         Piper   PA-23-250     Piper PA-23-250
13         Piper   PA-18-150     Piper PA-18-150
15        Cessna       R172E        Cessna R172E
2164


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
for col in ["Engine.Type", "Weather.Condition", "Purpose.of.flight", "Broad.phase.of.flight"]:
    df[col] = df[col].fillna("Unknown").astype(str).str.strip()
    df[col] = df[col].replace({"": "Unknown", "UNK": "Unknown", "Unk": "Unknown"})

df["Number.of.Engines.Clean"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce")

df[["Engine.Type", "Weather.Condition", "Number.of.Engines.Clean", "Purpose.of.flight", "Broad.phase.of.flight"]].head()

,Engine.Type,Weather.Condition,Number.of.Engines.Clean,Purpose.of.flight,Broad.phase.of.flight
1,Turbo Fan,VMC,4.0,Unknown,Taxi
2,Reciprocating,IMC,1.0,Personal,Cruise
3,Turbo Prop,VMC,2.0,Skydiving,Standing
7,Turbo Fan,VMC,3.0,Unknown,Taxi
8,Reciprocating,VMC,1.0,Personal,Climb


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
drop_cols = [c for c in df.columns if df[c].isna().mean() > 0.3]
df = df.drop(columns=drop_cols)
print(df.shape)
print(df.columns.tolist())

(17892, 38)
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Total.Passengers.Est', 'Serious.Fatal.Fraction', 'Fatal.Fraction', 'Aircraft.Damage.Clean', 'Destroyed.Flag', 'Make.Clean', 'Model.Clean', 'Plane.Type', 'Number.of.Engines.Clean']


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
df.to_csv("data/aviation_accidents_cleaned.csv", index=False)
print("Saved")

Saved
